In [3]:
import os
import cv2
import numpy as np
import albumentations as A
import random

In [ ]:

base_dir = r"C:/Users/athet/Downloads/archive/kaggle_3m"

tumour = []
nontumour = []

for patient in os.listdir(base_dir):
    patient_path = os.path.join(base_dir, patient)
    if not os.path.isdir(patient_path):
        continue

    for file in os.listdir(patient_path):
        if file.endswith(".tif") and not file.endswith("_mask.tif"):
            img_path = os.path.join(patient_path, file)
            mask_path = img_path.replace(".tif", "_mask.tif")

            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

            if np.sum(mask) > 0:
                tumour.append((img_path, mask_path))
            else:
                nontumour.append((img_path, mask_path))

print("Tumour slices     :", len(tumour))
print("Non-tumour slices :", len(nontumour))


Tumour slices     : 1373
Non-tumour slices : 2556


In [2]:
IMG_SIZE = 256

def preprocess_image(img):
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    return img.astype("float32") / 255.0

def preprocess_mask(mask):
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE))
    return (mask > 0).astype("float32")


In [4]:


augmentor = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.15, rotate_limit=15, p=0.7),
    A.ElasticTransform(p=0.3),
    A.RandomBrightnessContrast(p=0.3),
])


c:\Users\athet\mycnnenv\Lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [5]:
balanced_images = []
balanced_masks = []

# Add all non-tumour first
for img_path, mask_path in nontumour:
    img = cv2.imread(img_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    balanced_images.append(preprocess_image(img))
    balanced_masks.append(preprocess_mask(mask))

# Add original tumour slices
for img_path, mask_path in tumour:
    img = cv2.imread(img_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    balanced_images.append(preprocess_image(img))
    balanced_masks.append(preprocess_mask(mask))

# Now augment tumour until counts match


# Determine target = bigger class count
target = max(len(tumour), len(nontumour))

# Only augment tumour (minority class)
needed = target - len(tumour)

print("Augmenting tumour slices:", needed)


for _ in range(needed):
    img_path, mask_path = random.choice(tumour)

    img = cv2.imread(img_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    transformed = augmentor(image=img, mask=mask)

    balanced_images.append(preprocess_image(transformed["image"]))
    balanced_masks.append(preprocess_mask(transformed["mask"]))


Augmenting tumour slices: 1183


In [8]:
balanced_images = np.array(balanced_images)
balanced_masks  = np.array(balanced_masks)

print("Balanced dataset created.")
print("len of balanced images:", len(balanced_images))
print("len of balanced masks :", len(balanced_masks))
print("Final image shape:", balanced_images.shape)
print("Final mask shape :", balanced_masks.shape)


Balanced dataset created.
len of balanced images: 5112
len of balanced masks : 5112
Final image shape: (5112, 256, 256, 3)
Final mask shape : (5112, 256, 256)


In [7]:
np.savez_compressed("brainMRI_balanced.npz",
                    images=balanced_images,
                    masks=balanced_masks)
